# CoffeeGuard — Notebook 01: Chuẩn bị dữ liệu cho benchmark

**Chạy được cả trên laptop lẫn Colab.** Dữ liệu thô đã có sẵn trên laptop
(RoCoLe 2,2 GB đã giải nén), nên **chạy local là nhanh nhất** — không phải
upload gì cả. Đường Colab có sẵn để dùng khi cần.

Notebook này sinh ra ba thứ, và **chỉ ba thứ đó** đi tiếp sang notebook 02:

| Sinh ra | Là gì |
|---|---|
| `data/interim/rocole_index.csv` | nhãn + nhóm cây + khung bao lá + đa giác viền lá |
| `data/splits/rocole_folds.json` | cách chia 5 fold — **ĐÓNG BĂNG, phải commit** |
| `data/processed/<chế độ>/*.png` | ảnh 224×224 PNG không mất mát |

---

### Vì sao tiền xử lý làm một lần ở đây

Đồ án so sánh nhiều framework. Nếu để mỗi framework tự resize ảnh thì ta đang so
**thư viện resize** chứ không so kiến trúc:

| Nguồn sai lệch | Hậu quả |
|---|---|
| `tf.image.resize` mặc định `antialias=False`, PIL thì có | chênh 1–3% accuracy |
| `tf.io.decode_jpeg` (DCT nguyên) khác libjpeg của PIL | lệch vài giá trị pixel |

Cả hai đều **không gây lỗi** — chỉ làm một framework trông kém hơn. Nên ở đây
giải mã + cắt + thu nhỏ **một lần bằng PIL**, ghi **PNG không mất mát**, rồi mọi
framework đọc đúng cùng một mảng pixel.

### Notebook chỉ gọi + hiển thị, không tự định nghĩa lại gì

Thứ tự lớp, cách chia fold, cách cắt ảnh đều nằm trong `src/coffeeguard/bench/`.
Nếu notebook chép lại `CLASSES_6` thì có **hai bản định nghĩa**, và sớm muộn một
bản lệch — khi đó model train theo thứ tự này mà PWA giải mã theo thứ tự kia sẽ
cho **nhãn sai với confidence vẫn cao**, không exception, không log.
`tests/test_bench_contract.py` canh đúng chỗ đó.

## 0. Khởi động

In [ ]:
# --- O KHOI DONG: chay duoc CA tren Colab LAN tren laptop ---
import os, sys, json, time, zipfile, hashlib, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    TREN_COLAB = True
except ImportError:
    TREN_COLAB = False

if TREN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    GOC = Path("/content/CoffeeGuard")
    GOC.mkdir(parents=True, exist_ok=True)
    os.environ["COFFEEGUARD_ROOT"] = str(GOC)

    # Ma nguon hop dong (~40 KB). Tim theo thu tu: da giai nen > Drive > upload.
    if not (GOC / "src" / "coffeeguard" / "bench" / "contract.py").exists():
        nguon = Path("/content/drive/MyDrive/CoffeeGuard/coffeeguard_src.zip")
        if not nguon.exists():
            print("Chon file coffeeguard_src.zip (o §0 cua notebook 01 tren laptop sinh ra):")
            from google.colab import files
            up = files.upload()
            nguon = Path(next(iter(up)))
        with zipfile.ZipFile(nguon) as z:
            z.extractall(GOC)
        print("da giai nen ma nguon ->", GOC / "src")
    SRC = GOC / "src"
else:
    # Notebook nam o <repo>/notebooks/ nen goc repo la thu muc cha.
    GOC = Path.cwd()
    while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
        GOC = GOC.parent
    SRC = GOC / "src"

sys.path.insert(0, str(SRC))

from coffeeguard.bench import contract as ct
from coffeeguard.bench import paths as P

assert P.ROOT == GOC, f"Goc du an khong khop: {P.ROOT} != {GOC}"
print("moi truong :", "Google Colab" if TREN_COLAB else "laptop")
print("goc du an  :", P.ROOT)
print("classes_6  :", list(ct.CLASSES_6))

### 0b. Gói mã nguồn để mang lên Colab *(chỉ chạy khi đang ở laptop)*

Ô này tạo `data/coffeeguard_src.zip` (~40 KB). Chép nó vào
`Drive/MyDrive/CoffeeGuard/` thì ô §0 trên Colab tự tìm thấy.

In [ ]:
if not TREN_COLAB:
    src_cg = SRC / "coffeeguard"
    ma_nguon = [src_cg / "__init__.py", *sorted((src_cg / "bench").glob("*.py"))]
    dich = P.DATA / "coffeeguard_src.zip"
    dich.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(dich, "w", zipfile.ZIP_DEFLATED) as z:
        for p in ma_nguon:
            z.write(p, "src/" + str(p.relative_to(SRC)).replace("\\", "/"))
    print(f"{len(ma_nguon)} file -> {dich}  ({dich.stat().st_size/1024:.0f} KB)")
else:
    print("dang o Colab — bo qua")

## 1. Dữ liệu thô

Ba nguồn. Chỉ **RoCoLe** cần cho benchmark; BRACOL và symptom thuộc nhánh *mức
độ tổn thương* (xem `Docs/Ke_hoach_benchmark.md` §7) — cố ý **không** trộn vào
đây vì khác miền hoàn toàn với RoCoLe.

Trên Colab, ô này tải RoCoLe từ Mendeley. URL S3 của Mendeley **có chữ ký hết
hạn sau 300 giây**, nên mạng chậm thì không bao giờ tải xong — đó là lý do khâu
này hợp với Colab hơn là laptop.

In [ ]:
ROCOLE_ZIP_URL = "https://data.mendeley.com/public-api/zip/c5yvn32dzg/download/2"

def bam(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while blk := f.read(chunk):
            h.update(blk)
    return h.hexdigest()

def co_du_anh():
    return P.ROCOLE_PHOTOS.exists() and sum(1 for _ in P.ROCOLE_PHOTOS.glob("*.jpg")) >= 1560

if co_du_anh():
    print(f"[bo qua] da co {sum(1 for _ in P.ROCOLE_PHOTOS.glob('*.jpg'))} anh RoCoLe")
elif TREN_COLAB:
    zip_path = Path("/content/RoCoLe.zip")
    if not (zip_path.exists() and zip_path.stat().st_size > 2_000_000_000):
        print("[tai] RoCoLe ~2,09 GB ...")
        subprocess.run(["curl", "-L", "-C", "-", "--retry", "5", "--retry-all-errors",
                        "-o", str(zip_path), ROCOLE_ZIP_URL], check=True)
    print(f"   {zip_path.stat().st_size/1048576:.0f} MB — dang giai nen ...")
    dich = P.RAW / "rocole"
    dich.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        # Bo RoCoLe-voc.tar.gz (665 MB bounding box) — ta dung nhan phan loai.
        for n in z.namelist():
            if "voc.tar.gz" in n:
                continue
            # Lam phang: .../{Annotations,Photos}/x -> raw/rocole/{Annotations,Photos}/x
            phan = Path(n).parts
            if len(phan) >= 2 and phan[-2] in ("Annotations", "Photos"):
                ra = dich / phan[-2] / phan[-1]
                ra.parent.mkdir(parents=True, exist_ok=True)
                if not n.endswith("/"):
                    ra.write_bytes(z.read(n))
    print("   xong ->", dich)
else:
    raise FileNotFoundError(
        f"Khong thay anh RoCoLe o {P.ROCOLE_PHOTOS}.\n"
        "Giai nen file .zip o Downloads vao do, hoac chay notebook nay tren Colab."
    )

# Nhan da chuan hoa: neu chua co thi dung lai tu .xlsx trong Annotations.
if not P.ROCOLE_LABELS.exists():
    raise FileNotFoundError(
        f"Thieu {P.ROCOLE_LABELS}. Day la ban nhan da chuan hoa tu "
        "RoCoLe-classes.xlsx (cot: file, binary_label, multiclass_label, "
        "plant_id, leaf_id, face)."
    )
print("nhan  :", P.ROCOLE_LABELS)
print("mat na:", P.ROCOLE_COCO)

## 2. Bảng chỉ mục

Nối **ba nguồn độc lập**: nhãn (`.xlsx`), đa giác viền lá (`COCO`), file ảnh
thật. Ba nguồn lệch nhau thì hàm bên dưới **ném lỗi**, không tự bỏ qua — lặng lẽ
bỏ ảnh thiếu mặt nạ là cách chắc chắn nhất để bảng phân bố lớp trong báo cáo
khác với dữ liệu thực dùng.

In [ ]:
import pandas as pd
from coffeeguard.bench import index as bidx

pd.set_option("display.width", 200)

kq = bidx.dung_index()
df = kq.df
bidx.ghi_index(kq)

print(f"{len(df)} anh · {df.plant_id.nunique()} cay · {df.leaf_id.nunique()} la")
if kq.bo_qua["co_anh_khong_nhan"]:
    print("anh CO file nhung KHONG nhan (khong dung):", kq.bo_qua["co_anh_khong_nhan"])
print("sha256 index:", P.bam_file(P.INDEX_CSV)[:16], "...")

d6 = df.label6.value_counts().reindex(list(ct.CLASSES_6))
bang = pd.DataFrame({"n": d6, "%": (d6 / len(df) * 100).round(1),
                     "dai": [ct.band_of(c) for c in ct.CLASSES_6]})
display(bang)
print(f"-> doan 'healthy' moi luc da duoc {d6['healthy']/len(df)*100:.1f}% accuracy.")
print("   ACCURACY KHONG DUOC DUNG LAM CHI SO CHINH.")

### 2b. Hai giới hạn phải nêu trong báo cáo

Không phải thống kê cho vui — cả hai đều **im lặng**: model vẫn train, macro-F1
vẫn ra số, bảng so sánh vẫn đẹp.

In [ ]:
import collections

# --- 1. Nhan phu thuoc MAT LA ---
# Thang do MUC DO GI SAT — KHONG dung index trong CLASSES_6 lam thu hang:
# o do `red_spider_mite` chen vao giua healthy va rust_level_1, nen "healthy vs
# rust_level_1" bi tinh thanh lech 2 cap. Khoang cach van ra so, bang van ra
# dep, chi la phan loai sai kieu bat dong.
RANK_GI_SAT = {"healthy": 0, "rust_level_1": 1, "rust_level_2": 2,
               "rust_level_3": 3, "rust_level_4": 4}

g = df.groupby("leaf_id").label6.nunique()
tron = g[g > 1]
kieu = collections.Counter()
for lid, sub in df.groupby("leaf_id"):
    labs = set(sub.label6)
    if len(labs) == 1:
        kieu["hai mat giong nhau"] += 1
    elif "red_spider_mite" in labs and (labs - {"red_spider_mite"}):
        kieu["NHEN DO vs GI SAT/KHOE — mot la HAI van de"] += 1
    else:
        d = max(RANK_GI_SAT[c] for c in labs) - min(RANK_GI_SAT[c] for c in labs)
        kieu["lech 1 cap gi sat (that — not gi o mat duoi)" if d == 1
             else "lech 2-3 cap (nhieu nhan cua nguoi gan)"] += 1

print(f"la co nhan khac nhau giua hai mat: {len(tron)}/{len(g)} = {len(tron)/len(g)*100:.1f}%")
for k, v in kieu.most_common():
    print(f"   {v:4d} ({v/len(g)*100:4.1f}%)  {k}")
print("-> SAN SAI SO NOI TAI: voi nhung la nay khong model nao dung duoc ca hai mat")
print("   khi moi anh chi co mot nhan. Phan loai don nhan khong dien dat noi.")

# --- 2. Lop qua hiem ---
print("\nlop hiem:", {c: int(d6[c]) for c in ct.LOP_QUA_IT})
print("-> bao cao CHINH dung 5 dai; 4 cap chi la phu va phai ghi kem n.")

## 3. Chia 5 fold theo cây — đóng băng

* Theo **cây** chứ không theo ảnh: mỗi cây 2 lá × 2 mặt = 4 ảnh. Chia theo ảnh
  thì mặt còn lại của **chính chiếc lá đó** nằm ở tập test — rò rỉ gần tuyệt đối.
* **5 fold** chứ không một lần 70/15/15: chia một lần thì test chỉ có 3 ảnh
  `rust_level_4`; so 4 kiến trúc trên 3 ảnh là so nhiễu thuần. Với 5 fold mỗi
  ảnh vào test đúng một lần → **29 dự đoán** cho cấp 4.
* **Phân tầng**: chia ngẫu nhiên thuần dễ dồn hết cấp 4 vào 2–3 fold, khiến các
  fold còn lại không đo được gì cho lớp đó.
* Phương sai lấy từ **chênh lệch giữa 5 fold** → không cần chạy nhiều seed.

Ô này **từ chối ghi đè** file đã có. Sinh lại split nghĩa là mọi kết quả train
trước đó không còn so sánh được với sau đó.

In [ ]:
from coffeeguard.bench import splits as bsplit

GHI_DE_SPLIT = False   # doi True chi khi that su muon sinh lai TOAN BO

if P.FOLDS_JSON.exists() and not GHI_DE_SPLIT:
    meta = bsplit.doc_split()
    print(f"[bo qua] da co {P.FOLDS_JSON.name} (tao luc {meta['tao_luc']})")
    khop = meta.get("index_sha256") == P.bam_file(P.INDEX_CSV)
    print("index khop split:", "CO" if khop else "KHONG — xem canh bao ben duoi")
    if not khop:
        print("  !! Index da doi sau khi split duoc dong bang.")
        print("     Hoac khoi phuc index, hoac sinh lai split VA train lai TAT CA model.")
else:
    _, meta = bsplit.tao_split(df, index_sha256=P.bam_file(P.INDEX_CSV))
    bsplit.ghi_split(meta)
    print("da ghi", P.FOLDS_JSON, "-> COMMIT NGAY")

cot = [f"fold{i}" for i in range(meta["n_folds"])]
b6 = pd.DataFrame(meta["so_anh_moi_lop_theo_fold"], index=cot).T
b6["TONG"] = b6.sum(axis=1)
bB = pd.DataFrame(meta["so_anh_moi_dai_theo_fold"], index=cot).T
bB["TONG"] = bB.sum(axis=1)
print("\nso anh moi LOP khi fold do lam test:"); display(b6)
print("theo 5 DAI bao cao:"); display(bB)
for c in meta["canh_bao"]:
    print("[CANH BAO]", c)

## 4. Cắt + thu nhỏ ảnh

### Neo cạnh ô vuông vào cạnh nào — chỗ làm sai thì vẫn chạy trơn tru

Lá cà phê **dài**: khung bao trung vị 2,74:1. Nên ô vuông "bao trọn lá" (neo vào
cạnh **dài**) hoá ra lại chủ yếu là nền. Đo thật trên cả 1.559 ảnh:

| cách | % pixel là lá (p25 / trung vị / p75) |
|---|---|
| `letterbox` (chèn viền cho cả ảnh) | 10,8 / **14,2** / 18,3% |
| `leafsquare` neo cạnh **dài** 1,4× | 9,5 / **11,6** / 14,1% ← *tệ hơn cả letterbox* |
| `leafsquare` neo cạnh **ngắn** 1,4× | 56,1 / **60,9** / 63,5% |

**Cái giá phải trả:** neo cạnh ngắn **cắt mất hai đầu lá** — ô vuông chỉ dài
bằng ~0,5 chiều dài lá, nên vết bệnh sát chóp lá có thể không lọt khung. Đổi lại
mỗi vết bệnh còn trong khung được nhiều pixel hơn ~5 lần.

Đây là **đánh đổi**, không phải cải tiến thuần. Vì thế ô dưới sinh **cả hai** bộ
ảnh: `letterbox` là **ablation bắt buộc**. Không có nó thì "cửa sổ vuông giúp
ích" mới chỉ là một phép đếm pixel, chưa phải kết quả.

In [ ]:
from coffeeguard.bench import preprocess as bprep

BO_ANH = {}
for che_do in ("leafsquare", "letterbox"):
    kq_img = bprep.xu_ly(df, che_do=che_do, tien_do=lambda s: None)
    BO_ANH[che_do] = kq_img.dich
    t = kq_img.manifest["ti_le_pixel_la"]
    print(f"{che_do:12s} -> {kq_img.dich.name:26s} "
          f"ghi moi {kq_img.manifest['n_ghi_moi']:4d}  "
          f"la chiem {t['p25']*100:4.1f} / {t['trung_vi']*100:4.1f} / {t['p75']*100:4.1f} %")

### 4b. Nhìn tận mắt

Bảng số ở trên nói cửa sổ vuông giữ nhiều pixel lá hơn. Ô này cho **nhìn**, vì
số 60,9% không nói gì về việc vết bệnh có còn trong khung hay không.

Chọn cố ý: một lá khoẻ, một lá gỉ sắt cấp 1, một cấp 4, một nhện đỏ.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

chon = []
for lop in ("healthy", "rust_level_1", "rust_level_4", "red_spider_mite"):
    sub = df[df.label6 == lop]
    if len(sub):
        chon.append(sub.iloc[len(sub) // 2])

fig, axes = plt.subplots(2, len(chon), figsize=(3.1 * len(chon), 6.4))
for j, r in enumerate(chon):
    for i, che_do in enumerate(("letterbox", "leafsquare")):
        ax = axes[i, j]
        ax.imshow(Image.open(BO_ANH[che_do] / r.png))
        ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(f"{r.label6}\n({r.face})", fontsize=9)
        if j == 0:
            ax.set_ylabel(che_do, fontsize=10)
fig.suptitle("Hang tren: letterbox (~14% la)   ·   Hang duoi: cua so vuong (~61% la)",
             fontsize=11)
fig.tight_layout(); plt.show()

## 5. Kiểm chứng đầu-cuối bằng một baseline tối giản

Trước khi cho các kiến trúc nặng vào, chứng minh **nhãn + split + ảnh** đã nối
đúng, và chốt **định dạng file dự đoán** mà mọi framework phải xuất ra.

Baseline: hồi quy logistic trên ảnh xám 32×32. Chạy vài giây. Kết quả sẽ **tệ** —
đó là điều mong đợi. Việc của nó là hai:

1. Chứng minh đường ống chạy thông từ ảnh tới file dự đoán hợp lệ.
2. Cho một **sàn** để so. Nếu học sâu không vượt xa con số này thì có gì đó sai,
   và biết điều đó *trước* khi tốn giờ GPU rẻ hơn nhiều so với biết sau.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from coffeeguard.bench import predictions as bpred
from coffeeguard.bench import metrics as bm

CHE_DO_BASELINE = "leafsquare"
lab = bsplit.gan_fold(df, meta)

def tai_xam(pngs, thu_muc, canh=32):
    X = np.zeros((len(pngs), canh * canh), dtype=np.float32)
    for i, p in enumerate(pngs):
        with Image.open(thu_muc / p) as im:
            X[i] = np.asarray(im.convert("L").resize((canh, canh)),
                              dtype=np.float32).ravel() / 255.0
    return X

X = tai_xam(lab.png.tolist(), BO_ANH[CHE_DO_BASELINE])
y = lab.label6.map({c: i for i, c in enumerate(ct.CLASSES_6)}).to_numpy()

manh = []
for f in range(meta["n_folds"]):
    tr, te = lab.fold.to_numpy() != f, lab.fold.to_numpy() == f
    clf = LogisticRegression(max_iter=300).fit(X[tr], y[tr])
    prob = clf.predict_proba(X[te])
    Pm = np.zeros((prob.shape[0], len(ct.CLASSES_6)), dtype=np.float64)
    for j, c in enumerate(clf.classes_):      # bu cot cho lop vang mat o train
        Pm[:, c] = prob[:, j]
    Pm /= Pm.sum(axis=1, keepdims=True)
    sub = lab[te][["png", "fold", "plant_id", "leaf_id", "face"]].copy()
    sub["y_true"] = lab[te].label6.to_numpy()
    sub["y_pred"] = [ct.CLASSES_6[i] for i in Pm.argmax(1)]
    for j, c in enumerate(ct.CLASSES_6):
        sub[f"p_{c}"] = Pm[:, j]
    manh.append(sub)

pred = pd.concat(manh, ignore_index=True)
try:
    p_out = bpred.ghi_du_doan(pred, framework="sklearn", arch="logreg_gray32",
                              run_id=f"baseline-{CHE_DO_BASELINE}")
    print("da ghi", p_out.name)
except bpred.LoiHopDong as e:
    print("[bo qua ghi]", e)

kq_bl = bm.danh_gia(pred, n_bootstrap=500)
print(f"\nmacro-F1 6 lop : {kq_bl.macro_f1_6:.4f}")
print(f"macro-F1 5 dai : {kq_bl.macro_f1_dai:.4f}  "
      f"CI95 [{kq_bl.ci_dai['ci_lo']}, {kq_bl.ci_dai['ci_hi']}]")
print(f"accuracy       : {kq_bl.accuracy:.4f}   (doan bua: {kq_bl.accuracy_da_doan_bua:.4f})")
print("\n-> Day la SAN. Hoc sau phai vuot XA con so nay, khong phai vuot mot chut.")
display(kq_bl.bang_lop)

## 6. Đóng gói để mang lên Colab

Gói giữ **nguyên cây thư mục của repo**, và mang theo cả mã nguồn hợp đồng. Nhờ
vậy trên Colab chỉ cần giải nén + đặt `COFFEEGUARD_ROOT` là mọi đường dẫn chạy
đúng y nguyên — không sinh ra một phiên bản đường dẫn thứ hai dành riêng cho Colab.

In [ ]:
CHE_DO_DONG_GOI = "leafsquare"    # doi thanh "letterbox" de dong goi bo ablation

thu_muc = BO_ANH[CHE_DO_DONG_GOI]
goi = P.DATA / f"coffeeguard_bench_{thu_muc.name}.zip"
src_cg = SRC / "coffeeguard"
ma_nguon = [src_cg / "__init__.py", *sorted((src_cg / "bench").glob("*.py"))]

pngs = sorted(thu_muc.glob("*.png"))
with zipfile.ZipFile(goi, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in pngs:
        z.write(p, f"data/processed/{thu_muc.name}/{p.name}")
    z.write(thu_muc / "manifest.json", f"data/processed/{thu_muc.name}/manifest.json")
    z.write(P.INDEX_CSV, "data/interim/rocole_index.csv")
    z.write(P.FOLDS_JSON, "data/splits/rocole_folds.json")
    for p in ma_nguon:
        z.write(p, "src/" + str(p.relative_to(SRC)).replace("\\", "/"))

h = P.bam_file(goi)
(P.DATA / (goi.name + ".sha256")).write_text(f"{h}  {goi.name}\n", encoding="utf-8")
print(f"{len(pngs)} PNG + index + split + {len(ma_nguon)} file ma nguon")
print(f"-> {goi}  ({goi.stat().st_size/1048576:.1f} MB)")
print(f"sha256 {h}")
print("\nDAN CHUOI SHA256 TREN vao o cau hinh cua notebook 02.")
print("Notebook 02 se DUNG LAI neu khong khop — goi tai len hong giua chung la")
print("chuyen thuong, ma train tren goi hong thi moi con so deu vo nghia.")

## Việc tiếp theo

1. Chép `data/coffeeguard_bench_*.zip` **và** `data/coffeeguard_src.zip` lên
   `Drive/MyDrive/CoffeeGuard/`
2. **Commit `data/splits/rocole_folds.json`** — `.gitignore` đã mở ngoại lệ cho
   nó. Bốn model phải dùng chung đúng file này; nếu nó không nằm trong git thì
   máy khác sinh lại một cách chia khác, và bảng so sánh khi đó đang so nhiễu
   chứ không so kiến trúc.
3. Mở `02_benchmark_colab.ipynb` trên Colab (bật T4 GPU)

Kiểm lại bất cứ lúc nào:

```bash
python -m pytest tests/test_bench_contract.py -q
```